# 🏥 Breast Cancer Detection - Transfer Learning with TensorFlow

**Binary Classification: Benign vs Malignant Histopathology Images**

Built with KAPI blueprints - Production-ready deep learning pipeline.

---

## Dataset: Breast Histopathology Images
- **Source**: Kaggle (paultimothymooney/breast-histopathology-images)
- **Size**: 277K image patches (50x50 pixels)
- **Classes**: Benign (0), Malignant (1)
- **Task**: Binary classification using transfer learning

## Architecture:
- **Base Model**: MobileNetV2 (ImageNet pre-trained)
- **Training Strategy**: Two-phase (freeze → fine-tune)
- **Input Size**: 96x96x3
- **Data Augmentation**: Rotation, flips, shifts, zoom

## Step 1: Setup and Dataset Download

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Download dataset from Kaggle (requires kaggle.json credentials)
!kaggle datasets download -d paultimothymooney/breast-histopathology-images
!unzip -q breast-histopathology-images.zip -d breast_histopathology

print("✅ Dataset downloaded and extracted!")

## Step 2: Data Organization

Organize images into train/validation directories for ImageDataGenerator.

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

# Find all image paths and labels
data_dir = Path('breast_histopathology')
image_paths = []
labels = []

for patient_dir in data_dir.glob('*'):
    if patient_dir.is_dir():
        for class_dir in patient_dir.glob('*'):
            if class_dir.is_dir() and class_dir.name in ['0', '1']:
                label = int(class_dir.name)
                for img_path in class_dir.glob('*.png'):
                    image_paths.append(img_path)
                    labels.append(label)

print(f"📊 Total images: {len(image_paths):,}")
print(f"   Benign (0): {labels.count(0):,}")
print(f"   Malignant (1): {labels.count(1):,}")

# Split into train (80%) and validation (20%)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

# Organize into directories
organized_dir = Path('organized_data')
for split, paths, split_labels in [('train', train_paths, train_labels), ('validation', val_paths, val_labels)]:
    for label in [0, 1]:
        (organized_dir / split / str(label)).mkdir(parents=True, exist_ok=True)
    
    for path, label in zip(paths, split_labels):
        dest = organized_dir / split / str(label) / path.name
        if not dest.exists():
            shutil.copy(path, dest)

print(f"\n✅ Data organized:")
print(f"   Training: {len(train_paths):,} images")
print(f"   Validation: {len(val_paths):,} images")

## Step 3: Data Augmentation Setup

Aggressive augmentation to prevent overfitting on medical images.

In [ ]:
IMG_SIZE = 96
BATCH_SIZE = 32

# Training data: with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

# Validation data: only rescaling
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    organized_dir / 'train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    organized_dir / 'validation',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"✅ Data generators ready")
print(f"   Training batches: {len(train_generator)}")
print(f"   Validation batches: {len(val_generator)}")

## Step 4: Visualize Augmented Samples

In [ ]:
# Show augmented samples
sample_batch, sample_labels = next(train_generator)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i in range(8):
    axes[i].imshow(sample_batch[i])
    label_name = 'Malignant' if sample_labels[i] == 1 else 'Benign'
    axes[i].set_title(f'{label_name}', fontsize=12, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("✅ Augmented samples visualized")

## Step 5: Build Transfer Learning Model

Using MobileNetV2 pre-trained on ImageNet as feature extractor.

In [ ]:
# Load pre-trained MobileNetV2 (without top classification layer)
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model for initial training
base_model.trainable = False

# Build complete model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

model.summary()

## Step 6: Training Callbacks

Early stopping and learning rate reduction for optimal training.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_model.h5',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

print("✅ Callbacks configured")

## Step 7: Two-Phase Training

**Phase 1**: Train top layers with frozen base  
**Phase 2**: Fine-tune top layers of base model

In [ ]:
# Phase 1: Train top layers only
print("="*60)
print("PHASE 1: Training with frozen base model")
print("="*60)

history = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 1 complete!")

In [ ]:
# Phase 2: Fine-tune base model
print("\n" + "="*60)
print("PHASE 2: Fine-tuning top layers of base model")
print("="*60)

# Unfreeze last 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

history_fine = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 2 complete!")

## Step 8: Model Evaluation

Comprehensive metrics and visualizations.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns

print("📊 Evaluating model...\n")

# Get predictions
val_generator.reset()
y_pred_proba = model.predict(val_generator, verbose=1)
y_pred = (y_pred_proba > 0.5).astype(int)
y_true = val_generator.classes

# Classification report
print("\n📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Benign (0)', 'Malignant (1)']))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.title('Confusion Matrix - Cancer Detection', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Cancer Detection', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Model AUC Score: {roc_auc:.4f}")

In [ ]:
# Training history visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Phase 1 Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 0].set_title('Phase 1: Accuracy', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Phase 1 Loss
axes[0, 1].plot(history.history['loss'], label='Train', linewidth=2)
axes[0, 1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0, 1].set_title('Phase 1: Loss', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Phase 2 Accuracy
axes[1, 0].plot(history_fine.history['accuracy'], label='Train', linewidth=2)
axes[1, 0].plot(history_fine.history['val_accuracy'], label='Validation', linewidth=2)
axes[1, 0].set_title('Phase 2: Fine-tuning Accuracy', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Phase 2 Loss
axes[1, 1].plot(history_fine.history['loss'], label='Train', linewidth=2)
axes[1, 1].plot(history_fine.history['val_loss'], label='Validation', linewidth=2)
axes[1, 1].set_title('Phase 2: Fine-tuning Loss', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training history saved!")

## Step 9: Export Model for Production

In [ ]:
import json

# Save model
model_path = 'cancer_detection_model.h5'
model.save(model_path)

print(f"✅ Model saved: {model_path}")
print(f"📦 Size: {os.path.getsize(model_path) / (1024*1024):.2f} MB")

# Save metadata
model_info = {
    'model_name': 'MobileNetV2 Transfer Learning',
    'input_shape': [IMG_SIZE, IMG_SIZE, 3],
    'classes': ['Benign', 'Malignant'],
    'auc_score': float(roc_auc),
    'dataset': 'Breast Histopathology Images (Kaggle)',
    'total_params': model.count_params(),
    'preprocessing': {
        'resize': [IMG_SIZE, IMG_SIZE],
        'rescale': '1/255',
        'augmentation': ['rotation', 'flip', 'shift', 'zoom']
    }
}

with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("\n📝 Model metadata saved: model_info.json")
print(json.dumps(model_info, indent=2))

## Step 10: Test Predictions

In [ ]:
# Sample predictions visualization
val_generator.reset()
sample_images = []
sample_labels = []

for i in range(9):
    img_batch, label_batch = next(val_generator)
    sample_images.append(img_batch[0])
    sample_labels.append(label_batch[0])

predictions = model.predict(np.array(sample_images))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.ravel()

for idx in range(9):
    axes[idx].imshow(sample_images[idx])
    
    true_label = 'Malignant' if sample_labels[idx] == 1 else 'Benign'
    pred_label = 'Malignant' if predictions[idx] > 0.5 else 'Benign'
    confidence = predictions[idx][0] if predictions[idx] > 0.5 else 1 - predictions[idx][0]
    
    color = 'green' if true_label == pred_label else 'red'
    
    axes[idx].set_title(
        f'True: {true_label}\nPred: {pred_label} ({confidence:.1%})',
        fontsize=12,
        color=color,
        fontweight='bold'
    )
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Sample predictions visualized!")

## 🎉 Training Complete!

### Next Steps:

1. **Streamlit App**: Image upload UI for inference
2. **FastAPI Backend**: REST API for model serving
3. **Accuracy Dashboard**: Real-time metrics visualization

### Generated Files:
- `cancer_detection_model.h5` - Trained model (~15MB)
- `model_info.json` - Model metadata
- `confusion_matrix.png` - Evaluation metrics
- `roc_curve.png` - ROC curve
- `training_history.png` - Training progress
- `sample_predictions.png` - Test predictions

### ⚠️ Medical Disclaimer:
**This model is for educational purposes only.** Not for actual medical diagnosis. Always consult qualified healthcare professionals.

---

**Built with KAPI** - Production ML blueprints